# 🚀 Getting Started with the OSPool

Welcome! This notebook walks through the basics on how to use the **Open Science Pool (OSPool)** — a national computing resource that pools contributions from institutions across the United States. The OSPool is operated by the [OSG Consortium](https://osg-htc.org) through the [PATh project](https://path-cc.io), and it uses [HTCondor](https://htcondor.org) to manage jobs.

## ❔ About this environment
This notebook is powered by a mini-HTCondor pool running inside your Jupyter session and behaves like an OSPool Access Point (AP), **but isn't the real OSPool**. Use this pseudo-AP to test-drive HTCondor and the OSPool so you can see if this is the right solution for your computational work!

## 🌍 Overview of the OSPool

Below is a diagram that models how submitting jobs to the OSPool works.

<p style="text-align: center"><img src="https://portal.osg-htc.org/documentation/assets/overview_htcondor_job_submission.png" width=700px></p>

- You get an account on the **Access Point (AP)**. This is where you log in and prepare your software, files, and scripts.
- **HTCondor** is the job scheduling software that manages the jobs (a unit of work) you submit.
- Your jobs run on separate machines we call **Execution Points (EP)**. When they finish, outputs are sent back to you at the AP.

## 🖥️ Typical workflow

1. **Log in** to an OSPool *Access Point (AP)* — for example `ap40.uw.osg-htc.org`. You're logged in through a notebook right now!
2. **Upload or write your files** that you need to run your work, including scripts, data, and software.
4. **Describe your work** in an HTCondor *submit file*.
5. **Submit your work** with `condor_submit`, watch it with `condor_q`, and review your work after it runs.

Each grey cell below is a shell command. Click a cell and press **Shift+Enter** to run it.

## 0. Check that the pool is running

Everything in this notebook talks to an HTCondor pool. Let's make sure ours is up (and start it if it isn't).

In [1]:
# Ask the pool for a summary of its resources. If it doesn't answer, start the local pool and give it a few seconds.
condor_status -total 2>/dev/null || { echo "Pool not responding - starting the local HTCondor pool..."; condor_master && sleep 10 && condor_status -total; }


               Total Owner Claimed Unclaimed Matched Preempting  Drain Backfill BkIdle

  X86_64/LINUX 31737     0   28103      3618       0         16      0        0      0
 aarch64/LINUX     3     0       0         3       0          0      0        0      0

         Total 31740     0   28103      3621       0         16      0        0      0


## 1. Look around the Access Point

Let's look around the AP.

1. Check the version of HTCondor.

In [34]:
condor_version

$CondorVersion: 25.12.2 2026-07-20 BuildID: 933077 PackageID: 25.12.2-1+ubu24 $
$CondorPlatform: X86_64-Ubuntu_24.04 $


2. What machines/slots are available to run jobs?

   On the OSPool this lists many thousands of slots; our practice pool has just one (partitionable) slot on this machine.

In [ ]:
condor_status

3. What's in *your* job queue? (Empty for now.)

In [6]:
condor_q



-- Schedd: ap40.uw.osg-htc.org : <128.105.68.62:9618?... @ 09/11/26 20:23:04
OWNER BATCH_NAME      SUBMITTED   DONE   RUN    IDLE   HOLD  TOTAL JOB_IDS

Total for query: 0 jobs; 0 completed, 0 removed, 0 idle, 0 running, 0 held, 0 suspended 
Total for amber.lim@ap40.uw.osg-htc.org: 0 jobs; 0 completed, 0 removed, 0 idle, 0 running, 0 held, 0 suspended 
Total for all users: 40252 jobs; 30 completed, 12 removed, 26866 idle, 4539 running, 8805 held, 0 suspended



## 2. Describe our jobs

Let's run some work! For this notebook, we are going to create a job that uses a Python script to count words in text files.

### Our script: `wordcount.py`

Let's test our script. It should give us a summary of the word count task.

In [8]:
./wordcount.py Alice_in_Wonderland.txt

Analyzing Alice_in_Wonderland.txt
|  Total words = 29459
| Unique words = 6014
Elapsed time = 0.026443 secs


It also creates a `counts.Alice_in_Wonderland.tsv` file with a detailed table. Let's check the first few lines of that file.

In [17]:
head counts.Alice_in_Wonderland.tsv

the	1664
and	780
to	773
a	662
of	596
she	484
said	416
in	401
it	356
was	329


> Normally, we would **not** test this script on the AP, because most researcher jobs are computationally heavy! However, our word count example is light, so we are testing it here on the AP to show you what we expect for our outputs.

Now, let's imagine we want to run this script for all the books we have:
- Alice_in_Wonderland.txt
- Huckleberry_Finn.txt
- Pride_and_Prejudice.txt

Instead of running this one-by-one, we're going to leverage the OSPool and *submit all of this work at once*! Let's list all these books into a text file called `book.list` — we will use this later!

In [10]:
ls *.txt > book.list && cat book.list

Alice_in_Wonderland.txt
Huckleberry_Finn.txt
Pride_and_Prejudice.txt


Next, we will need a **submit file**, which translates the work we want to run into format that HTCondor understands.

### The submit file

Let's take a look at the submit file.

In [11]:
cat wordcount.sub

shell = ./wordcount.py $(book)
container_image = osdf://osg-public/containers/htc-images/ubuntu24.04.sif

log = log/job.$(Cluster).log
error = err/job.$(Cluster).$(Process).err
output = out/job.$(Cluster).$(Process).out

transfer_input_files = wordcount.py, $(book)

request_cpus   = 1
request_memory = 500MB
request_disk   = 1GB

queue book from book.list


## Submit file options

In our submit file, we use the following to describe our work.

| Submit file option | Purpose | Notes |
| --- | --- | --- |
| `shell` | The command you want to run. | More complex commands/scripts should be written in a wrapper script. |
| `container_image` | The container image that has our software environment installed. | For now, we're using a prebuilt container with Python installed. You can build and use your own containers on the OSPool. |
| `log` | A file created by HTCondor to record information about the job. |
| `error` | Where to save standard error messages. |
| `output` | Where to save standard output messages. |
| `transfer_input_files` | Files that we need to run our work. | The AP and the EPs that run your jobs do **not** share a filesystem, so list every file your job needs. |
| `request_*` | The resources (CPUs, memory, disk) we need to run our job. | GPUs may also be specified. |
| `queue` | How many jobs we want to run. Always at the end of the submit file. | This example uses the `queue <var> from <list>` syntax. |

> For more information about variables (i.e., `$(Cluster)`, `$(Process)`), see [Appendix](#Appendix).

## 3. Submit our jobs

Our submit file is ready to go! Let's submit the job:

In [12]:
condor_submit wordcount.sub

Submitting job(s)...
3 job(s) submitted to cluster 15295337.


## 4. Get the status of your jobs

Let's query the status of our job(s):

In [13]:
condor_q



-- Schedd: ap40.uw.osg-htc.org : <128.105.68.62:9618?... @ 09/11/26 21:18:34
OWNER     BATCH_NAME     SUBMITTED   DONE   RUN    IDLE  TOTAL JOB_IDS
amber.lim ID: 1529533   9/11 21:18      _      _      3      3 15295337.0-2

Total for query: 3 jobs; 0 completed, 0 removed, 3 idle, 0 running, 0 held, 0 suspended 
Total for amber.lim@ap40.uw.osg-htc.org: 3 jobs; 0 completed, 0 removed, 3 idle, 0 running, 0 held, 0 suspended 
Total for all users: 37918 jobs; 29 completed, 3 removed, 23379 idle, 5685 running, 8822 held, 0 suspended



You can also try the `-nobatch` option to see one job per line.

In [15]:
condor_q -nobatch



-- Schedd: ap40.uw.osg-htc.org : <128.105.68.62:9618?... @ 09/11/26 21:27:05
 ID      OWNER            SUBMITTED     RUN_TIME ST PRI SIZE CMD

Total for query: 0 jobs; 0 completed, 0 removed, 0 idle, 0 running, 0 held, 0 suspended 
Total for amber.lim@ap40.uw.osg-htc.org: 0 jobs; 0 completed, 0 removed, 0 idle, 0 running, 0 held, 0 suspended 
Total for all users: 45226 jobs; 28 completed, 3 removed, 30477 idle, 5897 running, 8821 held, 0 suspended



The jobs should complete within a few minutes!

> If you want a more "real-time" update of your jobs' status, open a Terminal tab, and use the command `condor_watch_q`. (It doesn't play nicely in the notebook interface, so don't run it in a notebook cell.)

## 5. Look at the results

When a job finishes, it leaves the queue, and `condor_q` will be empty. Let's check `condor_history` for a record of our jobs.

In [20]:
condor_history -limit 5

 ID          OWNER            SUBMITTED   RUN_TIME   ST   COMPLETED CMD
15295208.3119 siqi.mo         9/11 20:16   0+00:24:49 C   9/11 21:33 CA.sh 3119
15295208.2163 siqi.mo         9/11 20:16   0+00:38:46 C   9/11 21:33 CA.sh 2163
14999808.636  tanveer.islam   9/9  12:58   0+00:16:42 C   9/11 21:33 runner_osg
15295234.72   siqi.mo         9/11 21:06   0+00:05:42 C   9/11 21:33 MC.sh 256 
14999808.648  tanveer.islam   9/9  12:58   0+00:10:40 C   9/11 21:33 runner_osg


If our job ran successfully, we should have three `.tsv` files (one for each book) and three `.out` files with a printed summary. Let's run the commands below to confirm.

In [22]:
ls -lh counts.*.tsv

-rw-rw---- 1 amber.lim amber.lim  58K Sep 11 21:19 counts.Alice_in_Wonderland.tsv
-rw-r--r-- 1 amber.lim amber.lim 135K Sep 11 21:19 counts.Huckleberry_Finn.tsv
-rw-r--r-- 1 amber.lim amber.lim 145K Sep 11 21:19 counts.Pride_and_Prejudice.tsv


In [23]:
cat out/*.out

Analyzing Alice_in_Wonderland.txt
|  Total words = 29459
| Unique words = 6014
Elapsed time = 0.024251 secs
Analyzing Huckleberry_Finn.txt
|  Total words = 113342
| Unique words = 13831
Elapsed time = 0.097914 secs
Analyzing Pride_and_Prejudice.txt
|  Total words = 124580
| Unique words = 13644
Elapsed time = 0.036885 secs


If everything ran smoothly, the standard error files (specified by `error` in the submit file) should be empty. Let's check.

In [25]:
cat err/*.err

The `log` file is HTCondor's own record of the job's life — submitted, matched, executing, terminated, plus resource usage. The very end of this file shows a table about resources requested, allocated, and used. **This is useful for testing resource usage!** We recommend testing one or twojobs first and tailoring resource requests before scaling up to a full workload.

In [32]:
tail -n 20 log/*log

040 (15295337.001.000) 2026-09-11 16:19:29 Finished transferring output files
...
005 (15295337.001.000) 2026-09-11 16:19:29 Job terminated.
	(1) Normal termination (return value 0)
		Usr 0 00:00:00, Sys 0 00:00:00  -  Run Remote Usage
		Usr 0 00:00:00, Sys 0 00:00:00  -  Run Local Usage
		Usr 0 00:00:00, Sys 0 00:00:00  -  Total Remote Usage
		Usr 0 00:00:00, Sys 0 00:00:00  -  Total Local Usage
	138272  -  Run Bytes Sent By Job
	679892122  -  Run Bytes Received By Job
	138272  -  Total Bytes Sent By Job
	679892122  -  Total Bytes Received By Job
	Partitionable Resources :    Usage  Request Allocated 
	   Cpus                 :        0        1         1 
	   Disk (KB)            :   664231  1048576   1048576 
	   GPUs                 :                           0 
	   Memory (MB)          :        6      500       512 
	   TimeExecute (s)      :        1                    
	   TimeSlotBusy (s)     :       25                    
...


## 7. Where to go next

You just did the whole loop: write, submit, monitor, collect, scale. On the OSPool the steps are identical — only the size of the pool changes.

* **Get an OSPool account:** [portal.osg-htc.org/application](https://portal.osg-htc.org/application)
* **OSPool documentation:** [portal.osg-htc.org/documentation](https://portal.osg-htc.org/documentation/) — start with the [Quickstart](https://portal.osg-htc.org/documentation/htc_workloads/submitting_workloads/tutorial-quickstart/) and the [Roadmap to HTC Workload Submission](https://portal.osg-htc.org/documentation/htc_workloads/workload_planning/roadmap/)
* **HTCondor manual:** [htcondor.readthedocs.io](https://htcondor.readthedocs.io/)
* **Get help:** email [support@osg-htc.org](mailto:support@osg-htc.org) or drop by [OSPool office hours](https://portal.osg-htc.org/documentation/support_and_training/support/getting-help-from-RCFs/)

## Clean up (optional)

Remove the files this notebook created so you can run it again from a clean slate.

In [33]:
rm -rf log err out counts.*.tsv book.list

# Appendix

### Variables: `$(Cluster)`, `$(Process)`, and more

Any time you see this `$(variable)` syntax, this is HTCondor's variable syntax. Some have default values, like `$(Cluster)` and `$(Process)`, but you can also define your own.

| Variables | Purpose |
| --- | --- |
| `$(Cluster)` | An automatically generated unique ID per submission. |
| `$(Process)` | An automatically generated ID for jobs in each submission. i.e., if one submission has 3 jobs, `$(Process)` goes from 0-2. |
| `$(book)` | A custom variable we made for this example. |

### Where is "Alice_in_Wonderland.txt" in the submit file?

Recall the `book.list` file we made — this lists "Alice_in_Wonderland.txt", as well as other books we want to analyze.

In the `queue` statement, we used the `queue <var> from <list>` syntax.

What this does:
1. HTCondor will look for the `book.list` file.
2. HTCondor reads each line and assign its value to the variable `$(book)`. Each line corresponds to a unique job.
3. In the submit file, HTCondor replaces `$(book)` with its value for that job.

Essentially, HTCondor uses and handles a "for" loop to submit multiple jobs.

**As a result, we're able to analyze multiple books using one submit command and file instead of having to write many submit files!**